In [0]:
# Filename: src/dlt_pipelines/streaming_pipeline.py
import dlt
from pyspark.sql.functions import col, to_timestamp

@dlt.table(name="events_cleaned")
def events_cleaned():
    # Reading from the path confirmed in your catalog image
    return (spark.readStream.option("skipChangeCommits", "true")
            .table("workspace.bronze_layer.events_raw") 
            .withColumn("event_time", to_timestamp(col("event_time")))
            .dropDuplicates(["user_id", "event_time", "product_id"]))
    
    # --- QUARANTINE LAYER DATA ---
@dlt.table(
    name="events_quarantine",
    comment="Records that failed quality checks for manual review"
)
def events_quarantine():
    # Capture the opposite of your 'Clean' expectations
    return (
        dlt.read_stream("events_raw")
        .filter("(price <= 0) OR (user_id IS NULL) OR (event_time IS NULL)")
        .withColumn("ingestion_timestamp", current_timestamp())
    )